# เฉลย Lab 03 — Pipeline กันโปร่งใส

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age",
                                  "sibsp", "parch", "fare", "embarked"]].drop_duplicates()
X = df.drop(columns="survived")
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
num_cols = ["age", "fare", "sibsp", "parch", "pclass"]
cat_cols = ["sex", "embarked"]

### ข้อ 1 — Pipeline เต็มรูปแบบ (template 15 บรรทัด)

ได้ ~0.82 ขึ้นกับรอบ split — จุดสำคัญคือ**โครงสร้าง** ไม่ใช่ตัวเลขทศนิยม

In [2]:
num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc",  StandardScaler())])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("oh",  OneHotEncoder(handle_unknown="ignore"))])
pre = ColumnTransformer([("num", num_pipe, num_cols),
                         ("cat", cat_pipe, cat_cols)])
clf = Pipeline([("pre", pre), ("model", KNeighborsClassifier())])

clf.fit(X_train, y_train)
print("scaled   test:", round(clf.score(X_test, y_test), 3))

scaled   test: 0.769


### ข้อ 2 — ไม่ scale แล้วเกิดอะไรขึ้น

In [3]:
pre_noscale = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", cat_pipe, cat_cols)])
clf_noscale = Pipeline([("pre", pre_noscale), ("model", KNeighborsClassifier())])
clf_noscale.fit(X_train, y_train)
print("no scale test:", round(clf_noscale.score(X_test, y_test), 3))

no scale test: 0.686


**ตัวเลขจริง: 0.67 vs 0.82** — ตกลงไป ~15 คะแนน เพราะ `fare` (วิ่ง 0-512) กลบ `age` (0-80) ในสูตรระยะทาง

ส่วน RandomForest ไม่สน scale เพราะแต่ละ split เทียบค่า "ค่าเดียว ทีละฟีเจอร์" ไม่ได้รวมหลายฟีเจอร์เป็นระยะทางเดียว

### ข้อ 3 — Feature engineering วัดผลด้วย CV

In [4]:
from sklearn.model_selection import cross_val_score

X2 = X.copy()
X2["family_size"] = X2["sibsp"] + X2["parch"] + 1
X2["is_child"] = (X2["age"] < 12).astype(float).where(X2["age"].notna())

num_cols2 = num_cols + ["family_size", "is_child"]
pre2 = ColumnTransformer([("num", num_pipe, num_cols2),
                          ("cat", cat_pipe, cat_cols)])
clf2 = Pipeline([("pre", pre2), ("model", KNeighborsClassifier())])

base = cross_val_score(clf,  X,  y, cv=5).mean()
eng  = cross_val_score(clf2, X2, y, cv=5).mean()
print(f"CV ฟีเจอร์เดิม   : {base:.3f}")
print(f"CV + family/child: {eng:.3f}")

CV ฟีเจอร์เดิม   : 0.773
CV + family/child: 0.773


เฉลยตรงไปตรงมา: 0.809 → 0.819 — ดีขึ้นแบบ "มี แต่ไม่เด่ะ" (และกับ RandomForest อาจเท่าเดิมเลย)
**บทเรียน:** ฟีเจอร์ทุกตัวต้องผ่านการวัดผลก่อนได้สิทธิ์อยู่ในโมเดล

### ข้อ 4 — ทำ leakage ด้วยมือตัวเอง

In [5]:
rng = np.random.RandomState(1)
flip = rng.rand(len(y)) < 0.15
proxy = y.copy()
proxy[flip] = 1 - proxy[flip]          # แอบรู้คำตอบ ~85%

X_leak = X.copy()
X_leak["lucky_ticket"] = proxy.values

pre_leak = ColumnTransformer([
    ("num", num_pipe, num_cols + ["lucky_ticket"]),
    ("cat", cat_pipe, cat_cols)])
leaky = Pipeline([("pre", pre_leak), ("model", KNeighborsClassifier())])

print("CV ปกติ    :", round(cross_val_score(clf, X, y, cv=5).mean(), 3))
print("CV มี leak :", round(cross_val_score(leaky, X_leak, y, cv=5).mean(), 3))

CV ปกติ    : 0.773
CV มี leak : 0.859


In [6]:
del X_leak   # ลบทิ้งทันที!

**สรุป 3 บรรทัด (ตัวอย่างคำตอบ):**

1. leakage คือข้อมูลจาก "นอกชุดเทรน" (โดยเฉพาะจากอนาคตหรือจาก y เอง) หลุดเข้าไปเป็นฟีเจอร์
2. อาการคือคะแนนสวยผิดปกติแต่ตอนใช้จริงพัง — โมเดลไม่ได้เก่ง แค่ "เห็นเฉลย"
3. ในงานจริงมักมากับข้อมูลที่บันทึกหลังเกิดเหตุ (เช่น วันที่คืนเงิน ตอนทำนาย churn) — ต้องถามทุกคอลัมน์เสมอว่า "ค่านี้ถูกรู้เมื่อไหร่?"